# Schema Pruning Demo (BIRD) - V2 (Full Schema / No Pruning)
Notebook ini memakai schema utuh (tanpa pruning).
Retriever hanya opsional untuk debug skor (tidak mempengaruhi hasil).

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src").exists() and (parent / "Dataset").exists():
            PROJECT_ROOT = parent
            break

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pruning.schema_loader import SchemaLoader
from pruning.embedding_retriever_v2 import EmbeddingRetrieverV2
from pruning.evaluation import extract_gold_columns, precision_recall_f1_f6
from bird_datasets.bird_dataset_loader import BirdDatasetLoader

DATA_ROOT = PROJECT_ROOT / 'Dataset' / 'BIRD' / 'dev'
DATASET_PATH = DATA_ROOT / 'dev.json'
TABLES_PATH = DATA_ROOT / 'dev_tables.json'
DB_ROOT = DATA_ROOT / 'dev_databases' / 'dev_databases'

print('Project root:', PROJECT_ROOT)
print('Tables path:', TABLES_PATH)
print('DB root:', DB_ROOT)

Project root: d:\Kuliah\Skripsi\Schema Pruning
Tables path: d:\Kuliah\Skripsi\Schema Pruning\Dataset\BIRD\dev\dev_tables.json
DB root: d:\Kuliah\Skripsi\Schema Pruning\Dataset\BIRD\dev\dev_databases\dev_databases


In [2]:
# Input manual user (edit langsung variabel di bawah ini)
db_id = 'california_schools'
question = 'What is the highest eligible free rate for K-12 students in the schools in Alameda County?'
gold_sql = (
    "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1"
)
evidence = ''
# top_k = 20 
# lexical_weight = 0.3  

In [3]:
# Pilih sample dari dev.json (opsional)
use_sample_id = False
sample_id = 0  # ganti sesuai kebutuhan
if use_sample_id:
    import json
    samples = json.load(open(DATASET_PATH, 'r', encoding='utf-8'))
    item = samples[sample_id]
    db_id = item['db_id']
    question = item['question']
    gold_sql = item.get('SQL', item.get('sql', ''))
    evidence = item.get('evidence', '')
    print('Loaded sample_id:', sample_id)
    print('db_id:', db_id)
    print('question:', question)

In [4]:
# Alternatif input interaktif (set True jika ingin pakai input())
use_interactive = False
if use_interactive:
    db_id = input('db_id: ').strip()
    question = input('question: ').strip()
    gold_sql = input('gold SQL (boleh kosong): ').strip()
    if gold_sql == '':
        gold_sql = ''
    evidence = input('evidence (boleh kosong): ').strip()
    top_k = int(input('top_k (int, tidak dipakai di full schema): ').strip())
    lexical_weight = float(input('lexical_weight (0-1, hanya untuk debug): ').strip())

In [5]:
schema_loader = SchemaLoader(TABLES_PATH, DB_ROOT)

schema = schema_loader.get_schema(db_id)
final_columns = [col.full_name for col in schema.columns]

gold_columns = extract_gold_columns(gold_sql or '', schema)
precision, recall, f1, f6 = precision_recall_f1_f6(final_columns, gold_columns)

print('Mode:', 'full schema (no pruning)')
print('Total columns:', len(schema.columns))
print('Final columns:', final_columns)
print('Gold columns:', sorted(gold_columns))
print(f'Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | F6: {f6:.4f}')

Mode: full schema (no pruning)
Total columns: 89
Final columns: ['frpm.CDSCode', 'frpm.Academic Year', 'frpm.County Code', 'frpm.District Code', 'frpm.School Code', 'frpm.County Name', 'frpm.District Name', 'frpm.School Name', 'frpm.District Type', 'frpm.School Type', 'frpm.Educational Option Type', 'frpm.NSLP Provision Status', 'frpm.Charter School (Y/N)', 'frpm.Charter School Number', 'frpm.Charter Funding Type', 'frpm.IRC', 'frpm.Low Grade', 'frpm.High Grade', 'frpm.Enrollment (K-12)', 'frpm.Free Meal Count (K-12)', 'frpm.Percent (%) Eligible Free (K-12)', 'frpm.FRPM Count (K-12)', 'frpm.Percent (%) Eligible FRPM (K-12)', 'frpm.Enrollment (Ages 5-17)', 'frpm.Free Meal Count (Ages 5-17)', 'frpm.Percent (%) Eligible Free (Ages 5-17)', 'frpm.FRPM Count (Ages 5-17)', 'frpm.Percent (%) Eligible FRPM (Ages 5-17)', 'frpm.2013-14 CALPADS Fall 1 Certification Status', 'satscores.cds', 'satscores.rtype', 'satscores.sname', 'satscores.dname', 'satscores.cname', 'satscores.enroll12', 'satscores

## Pengujian semua dataset
Cell ini mengevaluasi seluruh sample di DATASET_PATH (default: dev.json).
Mode full schema = tidak ada pruning.

In [6]:
from statistics import mean
from itertools import islice

dataset = BirdDatasetLoader(DATASET_PATH)
max_samples = None  # ganti int untuk uji cepat

precisions, recalls, f1s, f6s, reductions = [], [], [], [], []
full_recall_flags = []
per_sample = []

for idx, sample in enumerate(dataset.iter_samples()):
    if max_samples is not None and idx >= max_samples:
        break

    schema = schema_loader.get_schema(sample.db_id)
    final_columns = [col.full_name for col in schema.columns]
    gold_columns = extract_gold_columns(sample.sql, schema)

    precision, recall, f1, f6 = precision_recall_f1_f6(
        final_columns, gold_columns
    )

    total_cols = len(schema.columns)
    pruned_cols = len(final_columns)
    reduction = 1.0 - (pruned_cols / total_cols) if total_cols else 0.0

    is_full_recall = recall >= 0.99999

    per_sample.append({
        "question_id": sample.question_id,
        "db_id": sample.db_id,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f6": f6,
        "reduction_ratio": reduction,
        "total_1_recall": is_full_recall,
    })

    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    f6s.append(f6)
    reductions.append(reduction)
    full_recall_flags.append(is_full_recall)

if per_sample:
    total_full_recall = sum(full_recall_flags)

    print("Mode:", "full_schema")
    print("Samples:", len(per_sample))
    print(f"Mean Precision: {mean(precisions):.4f}")
    print(f"Mean Recall: {mean(recalls):.4f}")
    print(f"Mean F1: {mean(f1s):.4f}")
    print(f"Mean F6: {mean(f6s):.4f}")
    print(f"Mean Reduction Ratio: {mean(reductions):.4f}")
    print(f"Total samples with 100% recall: {total_full_recall} / {len(per_sample)}")

    aggregate = {
        "num_samples": len(per_sample),
        "top_k": "all",
        "mean_precision": mean(precisions),
        "mean_recall": mean(recalls),
        "mean_f1": mean(f1s),
        "mean_f6": mean(f6s),
        "mean_reduction_ratio": mean(reductions),
        "total_full_recall": total_full_recall,
        "method": "fullschema",
    }

    from utils.results_writer import write_metrics_txt
    metrics_dir = PROJECT_ROOT / "outputs" / "pruning stats"
    txt_path = metrics_dir / "aggregate_v2_fullschema_top_all.txt"
    write_metrics_txt(
        txt_path,
        aggregate,
        title="Benchmark Summary V2 (fullschema, top_k=all)",
    )
    print("Saved:", txt_path)
else:
    print("No samples evaluated.")

Mode: full_schema
Samples: 1534
Mean Precision: 0.0990
Mean Recall: 1.0000
Mean F1: 0.1664
Mean F6: 0.6798
Mean Reduction Ratio: 0.0000
Total samples with 100% recall: 1534 / 1534
Saved: d:\Kuliah\Skripsi\Schema Pruning\outputs\pruning stats\aggregate_v2_fullschema_top_all.txt


## Ekspor dataset JSON untuk LLM
Cell ini membuat dataset JSON/JSONL berisi pertanyaan + schema prompt (schema utuh) untuk inference LLM.
Disertakan gold_sql dan gold_columns untuk evaluasi/analisis.

In [7]:
# # Ekspor dataset untuk LLM (full schema)
# import json
# from pathlib import Path

# dataset = BirdDatasetLoader(DATASET_PATH)
# max_samples = None  # ganti int untuk uji cepat
# use_evidence_prompt = True

# SYSTEM_PROMPT = "You are an expert text-to-SQL assistant for SQLite. Use only the provided schema and question. Output only the final SQL query as plain text. Do not add explanations or formatting. Quote identifiers with backticks if they contain spaces or symbols."

# def quote_column_full_name(full_name):
#     table, col = full_name.split(".", 1)
#     return f"{table}.`{col}`"

# def build_schema_prompt(schema, final_columns):
#     final_set = set(final_columns)
#     pk_set = {c.full_name for c in schema.primary_keys}
#     table_to_cols = {}
#     # Preserve schema order by iterating schema.columns
#     for col in schema.columns:
#         if col.full_name in final_set:
#             table_to_cols.setdefault(col.table, []).append(col)

#     def q(name):
#         return f"`{name}`"

#     def col_type(col):
#         return col.col_type if col.col_type else "text"

#     fk_by_table = {}
#     for col_a, col_b in schema.foreign_keys:
#         if col_a.full_name in final_set and col_b.full_name in final_set:
#             fk_by_table.setdefault(col_a.table, []).append((col_a, col_b))

#     lines = []
#     for table in schema.tables:
#         if table not in table_to_cols:
#             continue
#         cols = table_to_cols[table]
#         pk_cols = [c for c in cols if c.full_name in pk_set]

#         lines.append(f"CREATE TABLE {q(table)} (")
#         col_lines = [f"  {q(c.name)} {col_type(c)}" for c in cols]

#         if pk_cols:
#             pk_list = ", ".join(q(c.name) for c in pk_cols)
#             col_lines.append(f"  PRIMARY KEY ({pk_list})")

#         for col_a, col_b in fk_by_table.get(table, []):
#             col_lines.append(
#                 f"  FOREIGN KEY ({q(col_a.name)}) REFERENCES {q(col_b.table)} ({q(col_b.name)})"
#             )

#         lines.append(",\n".join(col_lines))
#         lines.append(");")
#         lines.append("")

#     return "\n".join(lines).rstrip()
# rows = []
# for idx, sample in enumerate(dataset.iter_samples()):
#     if max_samples is not None and idx >= max_samples:
#         break
#     schema = schema_loader.get_schema(sample.db_id)
#     final_columns = [col.full_name for col in schema.columns]
#     schema_prompt = build_schema_prompt(schema, final_columns)
#     gold_columns = extract_gold_columns(sample.sql, schema)
#     gold_columns_raw = sorted(gold_columns)
#     gold_columns_quoted = [quote_column_full_name(c) for c in gold_columns_raw]

#     user_prompt = f"Database: {sample.db_id}\\n\\nSchema:\\n{schema_prompt}\\n\\n"
#     if use_evidence_prompt and sample.evidence:
#         user_prompt += f"Evidence:\\n{sample.evidence}\\n\\n"
#     user_prompt += f"Question: {sample.question}"
#     messages = [
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": user_prompt},
#     ]

#     rows.append({
#         "question_id": sample.question_id,
#         "db_id": sample.db_id,
#         "question": sample.question,
#         "evidence": sample.evidence,
#         "schema_prompt": schema_prompt,
#         "top_k": len(final_columns),
#         "lexical_weight": lexical_weight,
#         "retriever_backend": "none",
#         "pruning_mode": "full_schema",
#         "gold_sql": sample.sql,
#         "gold_columns": gold_columns_quoted,
#         "gold_columns_raw": gold_columns_raw,
#         "messages": messages,
#     })

# output_dir = PROJECT_ROOT / "outputs" / "dataset" / "full_schema"
# output_dir.mkdir(parents=True, exist_ok=True)
# json_path = output_dir / "llm_dataset_v2_fullschema.json"
# jsonl_path = output_dir / "llm_dataset_v2_fullschema.jsonl"

# with json_path.open("w", encoding="utf-8") as f:
#     json.dump(rows, f, ensure_ascii=False, indent=2)

# with jsonl_path.open("w", encoding="utf-8") as f:
#     for row in rows:
#         f.write(json.dumps(row, ensure_ascii=False) + "\\n")

# print("Saved:", json_path)
# print("Saved:", jsonl_path)
# print("Rows:", len(rows))

## Benchmark kecil (opsional)
Benchmark V2 khusus pruning. Untuk mode full schema, gunakan cell "Pengujian semua dataset".

In [8]:
run_benchmark = False
if run_benchmark:
    print("Benchmark V2 hanya untuk pruning. Gunakan cell Pengujian semua dataset untuk mode full schema.")